In [2]:
%load_ext autoreload
%autoreload 2

import scanpy as sc
import sys
import os
import anndata
import pandas as pd
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
import anndata
import seaborn as sns
import torch
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

print("CUDA Available: ", torch.cuda.is_available())
print("Number of GPUs: ", torch.cuda.device_count())

CUDA Available:  True
Number of GPUs:  1


In [ ]:
data = sc.read_h5ad("../../../original_data/plant_all_days.h5ad")

In [59]:
print(data)
group_map = {'Nodule (21 dpi)': 21, 'Nodule (12 dpi)': 12, 'Root': 0}
data.obs['Group'] = data.obs['Sample_time'].map(group_map)
# print(data.obs[['Sample_time', 'Group']].drop_duplicates())

AnnData object with n_obs × n_vars = 26712 × 39337
    obs: 'n_genes', 'n_counts', 'Cluster', 'leiden_R', 'Cell type', 'Sample_time'
    var: 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches', 'mean', 'std', 'SNF', 'means_ForPickMock'
    uns: 'Cell type_colors'
    obsm: 'X_scvi', 'X_umap'
    layers: 'empty', 'normalize_log', 'raw'
    obsp: 'connectivities', 'distances'


In [60]:
# select Group == 12
data_group12 = data[data.obs["Group"] == 12].copy()

print("Group 12 shape:", data_group12.shape)
print("Clusters in Group 12:", data_group12.obs["Cluster"].unique())

# separate by Cluster and save each one
for cluster in data_group12.obs["Cluster"].unique():
    data_cluster = data_group12[
        data_group12.obs["Cluster"] == cluster
    ].copy()

    print("Cluster:", cluster, "shape:", data_cluster.shape)
    print("Total cells:", data_cluster.shape[0])

    # shuffle indices
    total_indices = np.arange(data_cluster.shape[0])
    np.random.shuffle(total_indices)

    # 80% train, 20% test
    train_size = int(0.8 * data_cluster.shape[0])

    train_indices = total_indices[:train_size]
    test_indices = total_indices[train_size:]

    train_cluster = data_cluster[train_indices, :].copy()
    test_cluster = data_cluster[test_indices, :].copy()

    print("Train:", train_cluster.shape)
    print("Test:", test_cluster.shape)

    # save h5ad
    train_cluster.write_h5ad(
        f"input/temp/exp2_seperate_train_day12_cluster_{cluster}.h5ad"
    )

    test_cluster.write_h5ad(
        f"input/exp2_test_day12_cluster_{cluster}.h5ad"
    )

Group 12 shape: (8229, 39337)
Clusters in Group 12: ['1', '6', '3', '7', '2', ..., '12', '8', '10', '14', '13']
Length: 14
Categories (14, object): ['0', '1', '2', '3', ..., '10', '12', '13', '14']
Cluster: 1 shape: (1779, 39337)
Total cells: 1779
Train: (1423, 39337)
Test: (356, 39337)
Cluster: 6 shape: (171, 39337)
Total cells: 171
Train: (136, 39337)
Test: (35, 39337)
Cluster: 3 shape: (298, 39337)
Total cells: 298
Train: (238, 39337)
Test: (60, 39337)
Cluster: 7 shape: (164, 39337)
Total cells: 164
Train: (131, 39337)
Test: (33, 39337)
Cluster: 2 shape: (1437, 39337)
Total cells: 1437
Train: (1149, 39337)
Test: (288, 39337)
Cluster: 0 shape: (2026, 39337)
Total cells: 2026
Train: (1620, 39337)
Test: (406, 39337)
Cluster: 5 shape: (442, 39337)
Total cells: 442
Train: (353, 39337)
Test: (89, 39337)
Cluster: 4 shape: (699, 39337)
Total cells: 699
Train: (559, 39337)
Test: (140, 39337)
Cluster: 9 shape: (356, 39337)
Total cells: 356
Train: (284, 39337)
Test: (72, 39337)
Cluster: 12 sha

In [61]:
# select Group == 21
data_group21 = data[data.obs["Group"] == 21].copy()

print("Group 21 shape:", data_group21.shape)
print("Clusters in Group 21:", data_group21.obs["Cluster"].unique())

# separate by Cluster and save each one
for cluster in data_group21.obs["Cluster"].unique():
    data_cluster = data_group21[
        data_group21.obs["Cluster"] == cluster
    ].copy()
    print("Cluster:", cluster, "shape:", data_cluster.shape)
    print("Total cells:", data_cluster.shape[0])

    # shuffle indices
    total_indices = np.arange(data_cluster.shape[0])
    np.random.shuffle(total_indices)

    # 80% train, 20% test
    train_size = int(0.8 * data_cluster.shape[0])

    train_indices = total_indices[:train_size]
    test_indices = total_indices[train_size:]

    train_cluster = data_cluster[train_indices, :].copy()
    test_cluster = data_cluster[test_indices, :].copy()

    print("Train:", train_cluster.shape)
    print("Test:", test_cluster.shape)

    # save h5ad
    train_cluster.write_h5ad(
        f"input/temp/exp2_seperate_train_day21_cluster_{cluster}.h5ad"
    )

    test_cluster.write_h5ad(
        f"input/exp2_test_day21_cluster_{cluster}.h5ad"
    )

Group 21 shape: (12004, 39337)
Clusters in Group 21: ['2', '12', '9', '4', '0', ..., '6', '10', '3', '13', '14']
Length: 15
Categories (15, object): ['0', '1', '2', '3', ..., '11', '12', '13', '14']
Cluster: 2 shape: (1526, 39337)
Total cells: 1526
Train: (1220, 39337)
Test: (306, 39337)
Cluster: 12 shape: (252, 39337)
Total cells: 252
Train: (201, 39337)
Test: (51, 39337)
Cluster: 9 shape: (571, 39337)
Total cells: 571
Train: (456, 39337)
Test: (115, 39337)
Cluster: 4 shape: (1380, 39337)
Total cells: 1380
Train: (1104, 39337)
Test: (276, 39337)
Cluster: 0 shape: (2762, 39337)
Total cells: 2762
Train: (2209, 39337)
Test: (553, 39337)
Cluster: 11 shape: (554, 39337)
Total cells: 554
Train: (443, 39337)
Test: (111, 39337)
Cluster: 1 shape: (2070, 39337)
Total cells: 2070
Train: (1656, 39337)
Test: (414, 39337)
Cluster: 8 shape: (327, 39337)
Total cells: 327
Train: (261, 39337)
Test: (66, 39337)
Cluster: 7 shape: (1332, 39337)
Total cells: 1332
Train: (1065, 39337)
Test: (267, 39337)
Clu

In [62]:
print(len(data_group21.obs["Cluster"].unique()))

15


In [63]:

for cluster in data_group21.obs["Cluster"].unique():
    #since cluster 11 not exist in day12.
    if cluster == '11':
        continue
    train_data_part1 = sc.read_h5ad(f'input/temp/exp2_seperate_train_day12_cluster_{cluster}.h5ad')                                                                                    
    print(train_data_part1.obs['Group'].unique())
    train_data_part2 = sc.read_h5ad(f'input/temp/exp2_seperate_train_day21_cluster_{cluster}.h5ad')
    print(train_data_part2.obs['Group'].unique())
    combined = anndata.concat([train_data_part1, train_data_part2])
    print(combined.obs['Group'].unique())
    combined.write_h5ad(f'input/exp2_train_cluster_{cluster}_adata.h5ad')

[12]
Categories (1, int64): [12]
[21]
Categories (1, int64): [21]
[12, 21]
Categories (2, int64): [12, 21]
[12]
Categories (1, int64): [12]
[21]
Categories (1, int64): [21]
[12, 21]
Categories (2, int64): [12, 21]
[12]
Categories (1, int64): [12]
[21]
Categories (1, int64): [21]
[12, 21]
Categories (2, int64): [12, 21]
[12]
Categories (1, int64): [12]
[21]
Categories (1, int64): [21]
[12, 21]
Categories (2, int64): [12, 21]
[12]
Categories (1, int64): [12]
[21]
Categories (1, int64): [21]
[12, 21]
Categories (2, int64): [12, 21]
[12]
Categories (1, int64): [12]
[21]
Categories (1, int64): [21]
[12, 21]
Categories (2, int64): [12, 21]
[12]
Categories (1, int64): [12]
[21]
Categories (1, int64): [21]
[12, 21]
Categories (2, int64): [12, 21]
[12]
Categories (1, int64): [12]
[21]
Categories (1, int64): [21]
[12, 21]
Categories (2, int64): [12, 21]
[12]
Categories (1, int64): [12]
[21]
Categories (1, int64): [21]
[12, 21]
Categories (2, int64): [12, 21]
[12]
Categories (1, int64): [12]
[21]

In [5]:
for cluster in ['0','1','2','3','4','5','6','7','8','9','10','12','13','14']:
    test_data_part1 = sc.read_h5ad(f'exp2_test_day12_cluster_{cluster}.h5ad')   
    print("12 day Total cells:", cluster, test_data_part1.shape[0])

12 day Total cells: 0 406
12 day Total cells: 1 356
12 day Total cells: 2 288
12 day Total cells: 3 60
12 day Total cells: 4 140
12 day Total cells: 5 89
12 day Total cells: 6 35
12 day Total cells: 7 33
12 day Total cells: 8 53
12 day Total cells: 9 72
12 day Total cells: 10 49
12 day Total cells: 12 56
12 day Total cells: 13 5
12 day Total cells: 14 10


In [6]:
for cluster in ['0','1','2','3','4','5','6','7','8','9','10','12','13','14']:
    test_data_part2 = sc.read_h5ad(f'input/exp2_test_day21_cluster_{cluster}.h5ad')   
    print("21 day Total cells:", cluster, test_data_part2.shape[0])

21 day Total cells: 0 553
21 day Total cells: 1 414
21 day Total cells: 2 306
21 day Total cells: 3 53
21 day Total cells: 4 276
21 day Total cells: 5 106
21 day Total cells: 6 10
21 day Total cells: 7 267
21 day Total cells: 8 66
21 day Total cells: 9 115
21 day Total cells: 10 44
21 day Total cells: 12 51
21 day Total cells: 13 34
21 day Total cells: 14 2


In [7]:
for cluster in ['0','1','2','3','4','5','6','7','8','9','10','12','13','14']:
    test_data_part2 = sc.read_h5ad(f'exp2_train_cluster_{cluster}_adata.h5ad')   
    print("21 day Total cells:", cluster, test_data_part2.shape[0])

21 day Total cells: 0 3829
21 day Total cells: 1 3079
21 day Total cells: 2 2369
21 day Total cells: 3 447
21 day Total cells: 4 1663
21 day Total cells: 5 775
21 day Total cells: 6 173
21 day Total cells: 7 1196
21 day Total cells: 8 473
21 day Total cells: 9 740
21 day Total cells: 10 367
21 day Total cells: 12 421
21 day Total cells: 13 154
21 day Total cells: 14 44
